# 02 - Silver Layer

Reads Bronze, applies explicit data types, identifies keys, deduplicates products, and runs quality checks.

In [37]:
from pyspark.sql import functions as F
from pyspark.sql.types import LongType, IntegerType, DecimalType
from pyspark.sql.window import Window
import os

from setup import get_spark_session
from config import (
    BRONZE_LAYER,
    SILVER_LAYER,
    BRONZE_TABLES,
    SILVER_TABLES,
)

from utils import (
    show_title,
    read_table,
    save_table,
    parse_mixed_date,
    print_table_location
)

spark = get_spark_session("sales-assessment-silver")

In [38]:
# Load reusable quality functions
%run ./00_quality_checks.ipynb

In [39]:
show_title("Silver Layer - Reading Bronze tables")

bronze_sales_order_detail = read_table(spark, BRONZE_LAYER, BRONZE_TABLES["sales_order_detail"])
bronze_sales_order_header = read_table(spark, BRONZE_LAYER, BRONZE_TABLES["sales_order_header"])
bronze_products = read_table(spark, BRONZE_LAYER, BRONZE_TABLES["products"])


Silver Layer - Reading Bronze tables


In [40]:
show_title("Silver Layer - Applying explicit data types")

silver_sales_order_detail_raw_typed = (
    bronze_sales_order_detail
    .select(
        F.col("SalesOrderID").cast(LongType()).alias("SalesOrderID"),
        F.col("SalesOrderDetailID").cast(LongType()).alias("SalesOrderDetailID"),
        F.col("OrderQty").cast(IntegerType()).alias("OrderQty"),
        F.col("ProductID").cast(LongType()).alias("ProductID"),
        F.col("UnitPrice").cast(DecimalType(18, 4)).alias("UnitPrice"),
        F.col("UnitPriceDiscount").cast(DecimalType(18, 4)).alias("UnitPriceDiscount"),
    )
)

invalid_order_qty_count = (
    silver_sales_order_detail_raw_typed
    .filter(F.col("OrderQty") < 0)
    .count()
)

silver_sales_order_detail = (
    silver_sales_order_detail_raw_typed
    .filter(
        F.col("OrderQty").isNull()
        | (F.col("OrderQty") >= 0)
    )
)

print(f"Records removed because of negative OrderQty: {invalid_order_qty_count}")

silver_sales_order_header = (
    bronze_sales_order_header
    .select(
        F.col("SalesOrderID").cast(LongType()).alias("SalesOrderID"),
        parse_mixed_date("OrderDate").alias("OrderDate"),
        parse_mixed_date("ShipDate").alias("ShipDate"),
        F.col("CustomerID").cast(LongType()).alias("CustomerID"),
        F.col("SalesPersonID").cast(LongType()).alias("SalesPersonID"),
        F.col("Freight").cast(DecimalType(18, 4)).alias("Freight"),
    )
)

silver_products_typed = (
    bronze_products
    .select(
        F.col("ProductID").cast(LongType()).alias("ProductID"),
        F.col("ProductDesc").alias("ProductDesc"),
        F.col("ProductNumber").alias("ProductNumber"),
        F.col("Color").alias("Color"),
        F.col("ProductSubCategoryName").alias("ProductSubCategoryName"),
        F.col("ProductCategoryName").alias("ProductCategoryName"),
    )
)


Silver Layer - Applying explicit data types
Records removed because of negative OrderQty: 2


In [41]:
show_title("Silver Layer - Deduplicating ProductID")

# ProductID is expected to be the Product primary key.
# If the raw file contains duplicated ProductID rows, keep the most informative row.
product_dedup_window = (
    Window
    .partitionBy("ProductID")
    .orderBy(
        F.when(F.col("ProductCategoryName").isNotNull(), F.lit(1)).otherwise(F.lit(0)).desc(),
        F.when(F.col("Color").isNotNull(), F.lit(1)).otherwise(F.lit(0)).desc(),
        F.col("ProductSubCategoryName").asc_nulls_last(),
    )
)

silver_products = (
    silver_products_typed
    .withColumn("rn", F.row_number().over(product_dedup_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)


Silver Layer - Deduplicating ProductID


In [42]:
show_title("Silver Layer - Saving store Parquet tables")

save_table(silver_sales_order_detail, SILVER_LAYER, SILVER_TABLES["sales_order_detail"])
save_table(silver_sales_order_header, SILVER_LAYER, SILVER_TABLES["sales_order_header"])
save_table(silver_products_typed, SILVER_LAYER, SILVER_TABLES["products_typed"])
save_table(silver_products, SILVER_LAYER, SILVER_TABLES["products"])

for table_name in [
    SILVER_TABLES["sales_order_detail"],
    SILVER_TABLES["sales_order_header"],
    SILVER_TABLES["products_typed"],
    SILVER_TABLES["products"],
]:
    print_table_location(SILVER_LAYER, table_name)


Silver Layer - Saving store Parquet tables
silver.store_sales_order_detail -> c:\Users\bruna.martins\Downloads\tech_assessment\parquet_tables\silver\store_sales_order_detail
silver.store_sales_order_header -> c:\Users\bruna.martins\Downloads\tech_assessment\parquet_tables\silver\store_sales_order_header
silver.store_products_typed -> c:\Users\bruna.martins\Downloads\tech_assessment\parquet_tables\silver\store_products_typed
silver.store_products -> c:\Users\bruna.martins\Downloads\tech_assessment\parquet_tables\silver\store_products


In [43]:
show_title("Silver Layer - Primary and foreign keys")

print("Primary keys:")
print("- store_sales_order_detail: SalesOrderDetailID")
print("- store_sales_order_header: SalesOrderID")
print("- store_products: ProductID")

print("Foreign keys:")
print("- store_sales_order_detail.SalesOrderID -> store_sales_order_header.SalesOrderID")
print("- store_sales_order_detail.ProductID -> store_products.ProductID")


Silver Layer - Primary and foreign keys
Primary keys:
- store_sales_order_detail: SalesOrderDetailID
- store_sales_order_header: SalesOrderID
- store_products: ProductID
Foreign keys:
- store_sales_order_detail.SalesOrderID -> store_sales_order_header.SalesOrderID
- store_sales_order_detail.ProductID -> store_products.ProductID


In [44]:
show_title("Silver Layer - Quality Checks")

import json
import os
import shutil

quality_results = []

quality_results.append(
    check_primary_key_unique(
        silver_sales_order_detail,
        SILVER_LAYER,
        SILVER_TABLES["sales_order_detail"],
        "SalesOrderDetailID",
    )
)

quality_results.append(
    check_primary_key_unique(
        silver_sales_order_header,
        SILVER_LAYER,
        SILVER_TABLES["sales_order_header"],
        "SalesOrderID",
    )
)

quality_results.append(
    check_primary_key_unique(
        silver_products_typed,
        SILVER_LAYER,
        SILVER_TABLES["products_typed"],
        "ProductID",
        severity="WARNING",
    )
)

quality_results.append(
    check_primary_key_unique(
        silver_products,
        SILVER_LAYER,
        SILVER_TABLES["products"],
        "ProductID",
    )
)

quality_results.append(
    check_foreign_key_exists(
        silver_sales_order_detail,
        silver_sales_order_header,
        SILVER_LAYER,
        SILVER_TABLES["sales_order_detail"],
        "SalesOrderID",
        "SalesOrderID",
    )
)

quality_results.append(
    check_foreign_key_exists(
        silver_sales_order_detail,
        silver_products,
        SILVER_LAYER,
        SILVER_TABLES["sales_order_detail"],
        "ProductID",
        "ProductID",
    )
)

for col_name in [
    "SalesOrderID",
    "SalesOrderDetailID",
    "OrderQty",
    "ProductID",
    "UnitPrice",
    "UnitPriceDiscount",
]:
    quality_results.append(
        check_not_null(
            silver_sales_order_detail,
            SILVER_LAYER,
            SILVER_TABLES["sales_order_detail"],
            col_name,
        )
    )

for col_name in ["SalesOrderID", "OrderDate", "ShipDate"]:
    quality_results.append(
        check_not_null(
            silver_sales_order_header,
            SILVER_LAYER,
            SILVER_TABLES["sales_order_header"],
            col_name,
        )
    )

for col_name in ["OrderQty", "UnitPrice", "UnitPriceDiscount"]:
    quality_results.append(
        check_non_negative(
            silver_sales_order_detail,
            SILVER_LAYER,
            SILVER_TABLES["sales_order_detail"],
            col_name,
        )
    )

quality_results.append(
    check_non_negative(
        silver_sales_order_header,
        SILVER_LAYER,
        SILVER_TABLES["sales_order_header"],
        "Freight",
    )
)

quality_results.append(
    check_negative_date_interval(
        silver_sales_order_header,
        SILVER_LAYER,
        SILVER_TABLES["sales_order_header"],
        "OrderDate",
        "ShipDate",
    )
)

negative_dates_count = quarantine_negative_dates(
    silver_sales_order_header,
    SILVER_TABLES["sales_order_header"],
    start_date_col="OrderDate",
    end_date_col="ShipDate",
)

print(f"Negative date records quarantined from Silver header: {negative_dates_count}")

quality_report_json_path = "parquet_tables/_tmp/silver_quality_report_json"

if os.path.exists(quality_report_json_path):
    shutil.rmtree(quality_report_json_path)

os.makedirs(quality_report_json_path, exist_ok=True)

json_file_path = os.path.join(quality_report_json_path, "part-00000.json")

with open(json_file_path, "w", encoding="utf-8") as file:
    for result in quality_results:
        file.write(json.dumps(result, default=str) + "\n")

silver_quality_report = spark.read.json(quality_report_json_path)

critical_failures = sum(
    1
    for result in quality_results
    if result["severity"] == "CRITICAL" and result["status"] == "FAIL"
)

save_table(
    silver_quality_report,
    SILVER_LAYER,
    SILVER_TABLES["quality_report"],
)

silver_quality_report.orderBy(
    "severity",
    "status",
    "table_name",
    "check_name",
).show(truncate=False)

print(f"Critical failures: {critical_failures}")

if FAIL_ON_CRITICAL and critical_failures > 0:
    raise Exception(
        f"Data quality failed with {critical_failures} critical failure(s)."
    )


Silver Layer - Quality Checks
Negative date records quarantined from Silver header: 0
+-------------------------------------------------+------------+------+-------------------------------------------------------------------------------------+--------+------+------------------------+
|check_name                                       |failed_count|layer |rule_description                                                                     |severity|status|table_name              |
+-------------------------------------------------+------------+------+-------------------------------------------------------------------------------------+--------+------+------------------------+
|primary_key_unique__ProductID                    |0           |silver|ProductID must be unique and not null. Duplicates and nulls are not allowed.         |CRITICAL|PASS  |store_products          |
|foreign_key_exists__ProductID                    |0           |silver|Every ProductID must exist in the referenced t